In [8]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import statsmodels.formula.api as smf
from itertools import product
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay,average_precision_score
)
from itertools import combinations
from scipy import sparse
from threadpoolctl import threadpool_limits

SEED = 42
SAMPLE_FRACTION = 0.0005  # 0.05%
DATA_PATH = "datasets/USDOT_1987_2025_Flights_Dataset/flights_2008_2025_cleaned.parquet"

In [2]:
model_data = pd.read_parquet("model_data.parquet")

split = np.load("model_split.npz")

train_df = model_data.iloc[split["train_idx"]]
test_df = model_data.iloc[split["test_idx"]]

In [3]:
final_features = [
    "distance",
    "crs_dep_time_minutes",
    "scheduled_elapsed_time",
    "scheduled_arrival_minutes",
    "month",
    "year",
    "dayofweek",
    "airline",
    "origin_airport",
    "dest_airport_group"
]

# Reuse existing training and testing datasets
X_train = train_df[final_features].copy()
X_test = test_df[final_features].copy()

# Convert arrival delays to binary outcomes
y_train = (train_df["arrival_delay"] >= 15).astype(int)
y_test = (test_df["arrival_delay"] >= 15).astype(int)

# Check class distributions
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))

Training distribution:
arrival_delay
0    0.800312
1    0.199688
Name: proportion, dtype: float64

Testing distribution:
arrival_delay
0    0.806179
1    0.193821
Name: proportion, dtype: float64


In [6]:

numerical_features = [
    "distance",
    "crs_dep_time_minutes",
    "scheduled_elapsed_time",
    "scheduled_arrival_minutes"
]

categorical_features = [
    "month",
    "year",
    "dayofweek",
    "airline",
    "origin_airport",
    "dest_airport_group"
]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numerical_features),
    ("categories", OneHotEncoder(
        handle_unknown="ignore"
    ), categorical_features)
])

# Use decision_function for threshold-independent evaluation.
# No probability calibration needed for ROC-AUC or AP.
svm_models = {
    "Linear SVM": Pipeline([
        ("features", preprocessor),
        ("model", SVC(
            kernel="linear",
            C=1.0,
            random_state=SEED
        ))
    ]),

    "RBF SVM": Pipeline([
        ("features", preprocessor),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            random_state=SEED
        ))
    ])
}

# Fit models
for name, model in svm_models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")


Trained: Linear SVM
Trained: RBF SVM


In [9]:
svm_results = []

for name, model in svm_models.items():

    y_score = model.decision_function(X_test)
    y_pred = model.predict(X_test)

    svm_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_score),
        "Average Precision": average_precision_score(y_test, y_score)
    })

svm_comparison = pd.DataFrame(svm_results).set_index("Model")

display(svm_comparison.round(4))

,Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision
Model,,,,,,
Linear SVM,0.8062,0.0,0.0,0.0,0.5878,0.2453
RBF SVM,0.8062,0.0,0.0,0.0,0.5943,0.2606


In [11]:
from sklearn.base import clone

balanced_svm_models = {}

for name, model in svm_models.items():

    balanced_model = clone(model)

    balanced_model.set_params(
        model__class_weight="balanced"
    )

    balanced_model.fit(X_train, y_train)

    balanced_svm_models[f"Balanced {name}"] = balanced_model

In [12]:
results = []

for name, model in balanced_svm_models.items():

    y_pred = model.predict(X_test)
    y_score = model.decision_function(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_score),
        "Average Precision": average_precision_score(y_test, y_score)
    })

display(
    pd.DataFrame(results)
    .set_index("Model")
    .round(4)
)

,Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision
Model,,,,,,
Balanced Linear SVM,0.5828,0.2587,0.6176,0.3646,0.6262,0.2776
Balanced RBF SVM,0.6387,0.2752,0.5287,0.3620,0.6357,0.2962


In [14]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rbf_pipeline = clone(
    balanced_svm_models["Balanced RBF SVM"]
)

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", 0.01, 0.1]
}

search = GridSearchCV(
    estimator=rbf_pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best CV Average Precision:", search.best_score_)

best_rbf_svm = search.best_estimator_

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best parameters: {'model__C': 10, 'model__gamma': 0.01}
Best CV Average Precision: 0.30100161966475597


In [15]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

# Tuned model
y_pred = best_rbf_svm.predict(X_test)
y_score = best_rbf_svm.decision_function(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Delayed", "Delayed"],
    zero_division=0
))

print("ROC-AUC:", roc_auc_score(y_test, y_score))
print("Average Precision:", average_precision_score(y_test, y_score))

              precision    recall  f1-score   support

 Not Delayed       0.87      0.61      0.72      4775
     Delayed       0.27      0.61      0.38      1148

    accuracy                           0.61      5923
   macro avg       0.57      0.61      0.55      5923
weighted avg       0.75      0.61      0.65      5923

ROC-AUC: 0.6430553477935678
Average Precision: 0.2966727242033589


# Support Vector Machine (SVM) Classification

## 1. Introduction

Following our logistic regression analysis, we investigated whether **Support Vector Machines (SVMs)** could improve our ability to predict significant arrival delays using the same ten predeparture features.

Unlike logistic regression, which estimates a linear relationship between predictors and the log-odds of an outcome, SVM attempts to identify a decision boundary that maximizes the margin between classes.

We evaluated two SVM configurations:

- **Linear SVM:** Uses a linear decision boundary.
- **RBF SVM:** Uses a Radial Basis Function kernel to capture nonlinear relationships between predictors.

Our classification target remained:

$$
Y=
\begin{cases}
1 & \text{Arrival Delay}\geq15\text{ minutes}\\
0 & \text{Arrival Delay}<15\text{ minutes}
\end{cases}
$$

The dataset remained imbalanced, with approximately **80.62% non-delayed flights and 19.38% delayed flights**.

## 2. Addressing Underfitting and Overfitting

An important consideration when developing SVM models is balancing model complexity against generalization.

**Underfitting** occurs when a model is too simple to capture meaningful relationships in the data. **Overfitting** occurs when a model learns patterns specific to the training observations that do not generalize to unseen flights.

SVM provides several mechanisms for controlling these problems.

### Regularization Parameter (C)

The parameter \(C\) controls the trade-off between maximizing the classification margin and penalizing margin violations.

- **Smaller C:** Stronger regularization, allowing more margin violations and potentially reducing overfitting.
- **Larger C:** Weaker regularization, penalizing margin violations more strongly and potentially reducing underfitting.

However, excessively large values of \(C\) may cause the model to fit training observations too closely.

### RBF Kernel Parameter (Gamma)

The RBF kernel is defined as:

$$
K(x_i,x_j)=\exp(-\gamma\lVert x_i-x_j\rVert^2)
$$

The parameter \(\gamma\) determines how quickly similarity decreases with distance.

- **Smaller gamma:** Produces smoother decision boundaries and may reduce overfitting.
- **Larger gamma:** Produces more localized and complex decision boundaries, potentially capturing additional patterns but increasing overfitting risk.

### Cross-Validation

To balance these effects, we used five-fold stratified cross-validation to evaluate combinations of \(C\) and \(\gamma\).

We selected hyperparameters using **Average Precision**, since significant delays represent the minority class and our objective is to identify delayed flights reliably.

Cross-validation helps assess how well candidate configurations generalize across different subsets of training observations, rather than selecting parameters solely from training accuracy.

## 3. Initial SVM Results

Our first experiment trained Linear and RBF SVM models without class weighting.

| Model | Accuracy | Recall | F1 | ROC-AUC | Average Precision |
|---|---:|---:|---:|---:|---:|
| Linear SVM | 0.8062 | 0.0000 | 0.0000 | 0.5878 | 0.2453 |
| RBF SVM | 0.8062 | 0.0000 | 0.0000 | 0.5943 | 0.2606 |

Both models classified every test observation as non-delayed, achieving an accuracy of approximately 80.62% while failing to identify any significantly delayed flights.

This highlighted the influence of class imbalance on the default decision boundary.

Although neither model detected delayed flights at its default boundary, ROC-AUC values above 0.50 indicated some ability to distinguish between the two classes through their decision scores.

## 4. Introducing Class Weighting

To address the imbalance, we retrained both models using `class_weight="balanced"`.

This assigns greater importance to errors involving the minority delayed class during optimization.

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | Average Precision |
|---|---:|---:|---:|---:|---:|---:|
| Balanced Linear SVM | 0.5828 | 0.2587 | 0.6176 | 0.3646 | 0.6262 | 0.2776 |
| Balanced RBF SVM | 0.6387 | 0.2752 | 0.5287 | 0.3620 | 0.6357 | 0.2962 |

Introducing class weighting substantially improved recall for both models.

Balanced Linear SVM identified approximately 61.8% of delayed flights, while Balanced RBF SVM identified approximately 52.9%.

However, this improvement came at the expense of precision and overall accuracy, as more non-delayed flights were incorrectly classified as delayed.

Importantly, Balanced RBF SVM also improved ROC-AUC and Average Precision, suggesting that class weighting affected not only the classification boundary but also the model's ability to rank flights by delay risk.

## 5. Hyperparameter Optimization

We subsequently performed a grid search for Balanced RBF SVM using five-fold stratified cross-validation.

The search evaluated nine combinations of \(C\) and \(\gamma\), resulting in 45 cross-validation fits.

The best parameters were:

| Hyperparameter | Selected value |
|---|---:|
| C | **10** |
| Gamma | **0.01** |
| Best CV Average Precision | **0.3010** |

The selected \(C=10\) allows a greater penalty for margin violations than our initial \(C=1\), while \(\gamma=0.01\) determines the smoothness of the nonlinear decision boundary.

These values achieved the highest cross-validated Average Precision among the tested configurations, but they do not establish that the model is entirely free from underfitting or overfitting.

## 6. Tuned SVM Results

We evaluated the tuned Balanced RBF SVM on the held-out test dataset.

| Metric | Result |
|---|---:|
| Accuracy | 0.61 |
| Precision (Delayed) | 0.27 |
| Recall (Delayed) | 0.61 |
| F1-score (Delayed) | 0.38 |
| ROC-AUC | **0.6431** |
| Average Precision | **0.2967** |

The tuned model identified approximately 61% of significantly delayed flights.

Compared with the untuned Balanced RBF SVM, tuning improved recall, F1-score, and ROC-AUC. However, Average Precision improved only slightly, from 0.2962 to 0.2967.

### Comparison with Logistic Regression

| Model | ROC-AUC | Average Precision | F1 |
|---|---:|---:|---:|
| Logistic Regression (threshold 0.17) | 0.6282 | 0.2822 | 0.3620 |
| Balanced Linear SVM | 0.6262 | 0.2776 | 0.3646 |
| Balanced RBF SVM | 0.6357 | 0.2962 | 0.3620 |
| **Tuned Balanced RBF SVM** | **0.6431** | **0.2967** | **~0.38** |

The tuned RBF SVM achieved the highest ROC-AUC and Average Precision among the models evaluated.

Its ROC-AUC improved by approximately 0.015 compared with logistic regression, suggesting a modest benefit from the tuned nonlinear classifier.

However, the improvement remains small, and additional validation would be required to establish whether it consistently generalizes to future flights.

## 7. Discussion

Our SVM experiments demonstrated three important findings.

First, **class imbalance significantly influenced classification performance**. Without class weighting, both SVM configurations predicted only the majority class, producing high accuracy but zero recall for delayed flights.

Second, introducing balanced class weights substantially improved the identification of delayed flights, although this increased false positives.

Third, hyperparameter tuning improved the RBF model's discrimination performance. The combination of \(C=10\) and \(\gamma=0.01\) achieved the strongest ROC-AUC observed in our experiments.

Nevertheless, the relatively modest ROC-AUC of 0.6431 and Average Precision of 0.2967 indicate that substantial uncertainty remains in predicting significant delays.

The results suggest that nonlinear modelling can capture additional predictive information, but increasing model complexity alone does not resolve the limitations of our current predictors.

## 8. Next Steps

Our remaining SVM task is to optimize the classification threshold using out-of-fold decision scores, rather than relying exclusively on the default SVM decision boundary.

This will allow us to compare threshold-optimized SVM against logistic regression using consistent evaluation objectives.

We will then investigate whether incorporating **ORD weather observations available one hour before scheduled departure** improves prediction performance.

The weather-enhanced analysis will use chronological training, validation, and test partitions to assess whether models generalize to later flights and whether weather features provide predictive information beyond the existing scheduled flight characteristics.